In [52]:
import pyarrow.compute as pc
from rapidmatch import ControlMatcher, MatchConfig

# Load data: file path, pandas DataFrame, or PyArrow Table
# 1 = campaign target, 0 = not targeted
config = MatchConfig(
    match_vars=["avg_monthly_spend_pre", "balance_pre", "ibb_pre","segment"],
    treatment_col="is_target",
    id_col="cust_id",                     # optional business id
    weights={"ibb_pre": 1.5},         # optional per-variable weights
    n=1,                             # 1:1 matching (1:n supported)
    tolerance=0.8,                   # keep pairs at/above this strength quantile
    min_control_pool_size=5,
    n_bins=20,
    monitor_vars=["age"],         # optional: post-hoc balance checks
    js_threshold=0.10,               # flag if category mixes differ too much
    ks_threshold=0.05,     
    progress=True,
    n_workers = 4,
    max_candidates_per_target=50
)

result = ControlMatcher(config).fit_match(r"D:\RapidSampler\temp\RapidMatch\credit_card_campaign_4M.parquet")

print(f"Matched: {result.coverage_summary['pct_matched']:.1%}")
print(f"Strength cutoff: {result.cutoff:.3f}")

pipeline:   0%|          | 0/10 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

score:   0%|          | 0/10877 [00:00<?, ?it/s]

match:   0%|          | 0/29618129 [00:00<?, ?it/s]

Matched: 20.0%
Strength cutoff: 0.998


In [53]:
import duckdb

In [54]:
database = 'experiment.db'
con = duckdb.connect(database)

In [55]:
results = result.pairs

In [56]:
con.execute("CREATE OR REPLACE TABLE dataset AS SELECT * FROM credit_card_campaign_4M.parquet")
con.execute("CREATE OR REPLACE TABLE result AS SELECT * FROM results")


In [65]:
con.sql("""SELECT match_status, COUNT(*) AS count FROM result GROUP BY ALL""").df()

,match_status,count
0,unmatched,39
1,no_control_available,165
2,below_tolerance,479599
3,matched,119900


In [70]:
# Build a 1:1 PAIRED cohort: only rows whose match_status='matched'.
# result.pairs has ONE row per target (matched + below_tolerance + unmatched
# + no_control_available) — joining on target_id alone would pull ALL targets
# back in. Filter to matched pairs so Treatment n == Control n.
con.execute("""
CREATE OR REPLACE TABLE matched_dataset AS
SELECT 'Treatment' AS group_type, dataset.*
FROM result
JOIN dataset ON dataset.cust_id = result.target_id
WHERE result.match_status = 'matched'
UNION ALL
SELECT 'Control' AS group_type, dataset.*
FROM result
JOIN dataset ON dataset.cust_id = result.control_id
WHERE result.match_status = 'matched'
""")


In [71]:
con.sql("""

SELECT group_type,
       COUNT(*)                                          AS n,
       AVG(avg_monthly_spend_pre)  AS avg_monthly_spend_pre,
       AVG(balance_pre)            AS balance_pre,
       AVG(ibb_pre)                AS ibb_pre,
       AVG(avg_monthly_spend_post) AS avg_monthly_spend_post,
       AVG(balance_post)           AS balance_post,
       AVG(ibb_post)               AS ibb_post
FROM matched_dataset
GROUP BY group_type
ORDER BY group_type

""").df()


,group_type,n,avg_monthly_spend_pre,balance_pre,ibb_pre,avg_monthly_spend_post,balance_post,ibb_post
0,Control,119900,313.100320,419.442547,159.248699,322.992643,426.804057,162.016860
1,Treatment,119900,313.100934,419.436481,159.250108,355.214087,443.572883,168.362526
